# LeetCode 1801 — Number of Orders in the Backlog bằng Python

Link đề: https://leetcode.com/problems/number-of-orders-in-the-backlog/ · Độ khó: **Medium** · Chủ đề: Array, Heap (Priority Queue), Simulation

Skeleton LeetCode:

```python
class Solution:
    def getNumberOfBacklogOrders(self, orders: List[List[int]]) -> int:
        ...
```

Lộ trình: **mô phỏng từng lệnh đơn lẻ** (tách lô thành từng đơn vị — chỉ cho số lượng nhỏ) → **mô phỏng theo lô với danh sách + quét min/max** `O(n²)` → **hai heap theo lô** `O(n log n)` (lời giải nộp) → **hai heap sửa đỉnh tại chỗ** (bớt thao tác push/pop).

## 1. Mục tiêu học

1. Đọc hiểu một bài **mô phỏng sổ lệnh** (order book): lệnh mua khớp với lệnh bán **rẻ nhất**, lệnh bán khớp với lệnh mua **đắt nhất**.
2. Dùng `heapq` làm **min-heap** và mẹo **đổi dấu** để có **max-heap**.
3. Xử lý **theo lô** (`amount` tới `10⁹`) thay vì từng đơn vị — tách khớp một phần (partial fill) và đẩy phần dư lại heap.
4. Phân tích **amortized**: mỗi lô được đẩy vào heap tối đa một lần ⇒ tổng số lần pop bị chặn.
5. Áp dụng **modulo** đúng chỗ (chỉ khi tính tổng cuối), và kiểm chứng bằng mô phỏng từng đơn vị.

## 2. Đề bài & ví dụ

Cho mảng 2 chiều `orders`, mỗi `orders[i] = [price, amount, orderType]` nghĩa là có `amount` lệnh loại `orderType` với giá `price`:
- `orderType = 0`: lô lệnh **mua** (buy);
- `orderType = 1`: lô lệnh **bán** (sell).

Mỗi lô gồm `amount` lệnh độc lập cùng giá và loại. Mọi lệnh của `orders[i]` được đặt **trước** mọi lệnh của `orders[i+1]`.

**Backlog** (sổ tồn) ban đầu rỗng. Khi đặt một lệnh:
- Lệnh **mua** giá `p`: xem lệnh **bán có giá nhỏ nhất** trong backlog. Nếu giá đó `≤ p` thì hai lệnh khớp và lệnh bán bị xóa khỏi backlog; ngược lại lệnh mua được thêm vào backlog.
- Lệnh **bán** giá `p`: xem lệnh **mua có giá lớn nhất** trong backlog. Nếu giá đó `≥ p` thì khớp và lệnh mua bị xóa; ngược lại lệnh bán vào backlog.

Trả về **tổng số lệnh** trong backlog sau khi đặt hết, **modulo `10⁹ + 7`**.

**Ví dụ 1:** `orders = [[10,5,0],[15,2,1],[25,1,1],[30,4,0]]` → `6` (còn 5 lệnh mua giá 10 và 1 lệnh mua giá 30).

**Ví dụ 2:** `orders = [[7,1000000000,1],[15,3,0],[5,999999995,0],[5,1,1]]` → `999999984` (còn `10⁹−3` lệnh bán giá 7 và `999999995−1` lệnh mua giá 5; tổng `1999999991 mod (10⁹+7)`).

**Ràng buộc:** `1 ≤ orders.length ≤ 10⁵`, `1 ≤ price, amount ≤ 10⁹`, `orderType ∈ {0, 1}`.

In [1]:
import heapq
import random
import time
from typing import List

MOD = 10**9 + 7
examples = [
    ([[10, 5, 0], [15, 2, 1], [25, 1, 1], [30, 4, 0]], 6),
    ([[7, 1000000000, 1], [15, 3, 0], [5, 999999995, 0], [5, 1, 1]], 999999984),
]
for orders, want in examples:
    print(orders, "-> kỳ vọng", want)

[[10, 5, 0], [15, 2, 1], [25, 1, 1], [30, 4, 0]] -> kỳ vọng 6
[[7, 1000000000, 1], [15, 3, 0], [5, 999999995, 0], [5, 1, 1]] -> kỳ vọng 999999984


## 3. Kiến thức nền

### 3.1 `heapq` — min-heap trên list

`heapq.heappush(h, x)` và `heapq.heappop(h)` đều `O(log n)`; `h[0]` luôn là phần tử **nhỏ nhất**. Phần tử có thể là tuple — so sánh theo thứ tự từ điển (phần tử đầu trước).

In [2]:
h = []
for item in [(15, 2), (25, 1), (7, 9), (15, 1)]:
    heapq.heappush(h, item)
print("đỉnh:", h[0])
print("pop lần lượt:", [heapq.heappop(h) for _ in range(len(h))])

đỉnh: (7, 9)
pop lần lượt: [(7, 9), (15, 1), (15, 2), (25, 1)]


### 3.2 Max-heap bằng cách đổi dấu

Python không có max-heap sẵn. Lưu `(-price, amount)` thì phần tử nhỏ nhất ứng với **giá lớn nhất**. Khi đọc ra nhớ đổi dấu lại.

In [3]:
buy = []
for p, a in [(10, 5), (30, 1), (20, 2)]:
    heapq.heappush(buy, (-p, a))
neg_p, a = buy[0]
print("lệnh mua đắt nhất: giá", -neg_p, "số lượng", a)

lệnh mua đắt nhất: giá 30 số lượng 1


### 3.3 Sửa đỉnh heap mà không phá thứ tự

Nếu chỉ thay **số lượng** (thành phần thứ hai) của phần tử ở đỉnh mà **giữ nguyên giá**, thứ tự heap vẫn hợp lệ (vì giá là khóa so sánh chính, và các phần tử khác có giá ≥ nó). Tuple bất biến nên ta dùng list `[price, amount]` để sửa tại chỗ.

In [4]:
sell = [[7, 10], [9, 3], [8, 4]]
heapq.heapify(sell)
sell[0][1] -= 6            # khớp 6 đơn vị với lệnh bán rẻ nhất
print(sell[0], "| heap vẫn hợp lệ:", all(sell[(i - 1) // 2] <= sell[i] for i in range(1, len(sell))))

[7, 4] | heap vẫn hợp lệ: True


### 3.4 Modulo `10⁹ + 7`

Python có số nguyên lớn nên không tràn; đề yêu cầu trả **kết quả** modulo. **Không** được lấy modulo các số lượng giữa chừng vì chúng còn được **so sánh** (`a >= y`).

In [5]:
total = (10**9 - 3) + (999999995 - 1)
print(total, "->", total % MOD)

1999999991 -> 999999984


### 3.5 `min`/`max` với `key` trên list

Cách "ngây thơ" để tìm lệnh bán rẻ nhất: `min(range(len(L)), key=lambda i: L[i][0])` — `O(len(L))` mỗi lần.

In [6]:
L = [[15, 2], [7, 1], [25, 1]]
i = min(range(len(L)), key=lambda k: L[k][0])
print("chỉ số lệnh bán rẻ nhất:", i, L[i])

chỉ số lệnh bán rẻ nhất: 1 [7, 1]


## 4. Ý tưởng nền tảng / trực giác

Theo dõi Ví dụ 1 bằng tay. Ký hiệu backlog mua `B` và bán `S` (dạng `giá×số lượng`):

| lô | hành động | `B` | `S` |
|---|---|---|---|
| mua 10×5 | `S` rỗng → vào backlog | 10×5 | — |
| bán 15×2 | mua đắt nhất 10 < 15 → vào backlog | 10×5 | 15×2 |
| bán 25×1 | 10 < 25 → vào backlog | 10×5 | 15×2, 25×1 |
| mua 30×4 | bán rẻ nhất 15 ≤ 30: khớp 2; tiếp 25 ≤ 30: khớp 1; `S` rỗng → 1 vào backlog | 10×5, 30×1 | — |

Tổng `5 + 1 = 6`. ✔

Hai thao tác cần nhanh: "lấy lệnh bán **rẻ nhất**" và "lấy lệnh mua **đắt nhất**" ⇒ **hai hàng đợi ưu tiên**. Và vì `amount` tới `10⁹`, khớp **cả lô một lúc**: `k = min(còn lại, số lượng ở đỉnh)`.

## 5. Cách 1 — Brute Force / baseline: từng lệnh đơn lẻ, danh sách thường

**Ý tưởng:** tách mỗi lô thành `amount` lệnh riêng; backlog là hai list giá. Mỗi lệnh mua tìm `min(sell)` bằng quét; mỗi lệnh bán tìm `max(buy)`. Đúng theo đúng câu chữ của đề ⇒ chuẩn đối chiếu (chỉ dùng khi tổng `amount` nhỏ).

In [7]:
def backlog_brute(orders):
    buy, sell = [], []                    # mỗi phần tử là giá của MỘT lệnh
    for p, a, t in orders:
        for _ in range(a):
            if t == 0:
                if sell and min(sell) <= p:
                    sell.remove(min(sell))
                else:
                    buy.append(p)
            else:
                if buy and max(buy) >= p:
                    buy.remove(max(buy))
                else:
                    sell.append(p)
    return (len(buy) + len(sell)) % MOD

print(backlog_brute(examples[0][0]), "| kỳ vọng", examples[0][1])

6 | kỳ vọng 6


**Độ phức tạp:** gọi `T = Σ amount` (tới `10¹⁴`!). Mỗi lệnh quét backlog `O(T)` ⇒ `O(T²)`. Ví dụ 2 đã có `T ≈ 2·10⁹` — không chạy nổi.

**Hai nút thắt:** (1) xử lý **từng đơn vị** thay vì cả lô; (2) tìm min/max bằng **quét tuyến tính**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — khớp theo lô:** khi lô mua `(p, a)` gặp đỉnh bán `(q, y)` với `q ≤ p`, số lệnh khớp là `k = min(a, y)`. Nếu `a ≥ y`, lô bán biến mất (pop) và `a -= y`; nếu `a < y`, lô bán còn `y − a` (cập nhật đỉnh) và `a = 0`.

**Insight 2 — cấu trúc dữ liệu:** backlog bán cần **min theo giá** ⇒ min-heap `(price, amount)`; backlog mua cần **max theo giá** ⇒ min-heap `(-price, amount)`.

**Quy tắc "nếu… thì…" cho lô mua `(p, a)`:**
- *Nếu* `a > 0` và `sell` khác rỗng và `sell[0].price ≤ p` *thì* khớp theo Insight 1, lặp lại.
- *Nếu* hết vòng mà `a > 0` *thì* đẩy `(-p, a)` vào `buy`.

Lô bán đối xứng với `buy[0]` và điều kiện `−buy[0][0] ≥ p`.

**Invariant:** sau mỗi lô, **không** tồn tại cặp mua giá `b` và bán giá `s` trong backlog với `b ≥ s` (nếu có, chúng đã khớp). Nghĩa là `max(buy) < min(sell)` — hai heap "không chồng giá".

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Theo lô, backlog là list, quét min/max (`O(n²)`)

In [8]:
def backlog_batch_list(orders):
    buy, sell = [], []                    # mỗi phần tử [price, amount]
    for p, a, t in orders:
        if t == 0:
            while a and sell:
                i = min(range(len(sell)), key=lambda k: sell[k][0])
                if sell[i][0] > p:
                    break
                k = min(a, sell[i][1])
                a -= k; sell[i][1] -= k
                if sell[i][1] == 0:
                    sell.pop(i)
            if a:
                buy.append([p, a])
        else:
            while a and buy:
                i = max(range(len(buy)), key=lambda k: buy[k][0])
                if buy[i][0] < p:
                    break
                k = min(a, buy[i][1])
                a -= k; buy[i][1] -= k
                if buy[i][1] == 0:
                    buy.pop(i)
            if a:
                sell.append([p, a])
    return sum(x[1] for x in buy + sell) % MOD

for orders, want in examples:
    print(backlog_batch_list(orders), want)

6 6
999999984 999999984


**Độ phức tạp:** mỗi lần tìm min/max quét `O(m)` với `m ≤ n` lô trong backlog. Số lần tìm = số lần khớp + `n` (mỗi lô có tối đa một lần "dừng"). Số lần khớp làm **xóa** một lô ≤ `n`; số lần khớp **không** xóa ≤ 1 mỗi lô ⇒ tổng `O(n)` lần tìm ⇒ `O(n²)`. Với `n = 10⁵` là `10¹⁰` — quá chậm. Nút thắt: tìm min/max ⇒ dùng **heap**.

### 7.2 Cách 3 — Hai heap theo lô (`O(n log n)`) — lời giải chuẩn

Code demo in từng bước khớp:

In [9]:
def backlog_heaps_demo(orders):
    buy, sell = [], []                    # buy: (-price, amount), sell: (price, amount)
    for p, a, t in orders:
        kind = "MUA" if t == 0 else "BÁN"
        print(f"Lô {kind} giá {p} x{a}")
        if t == 0:
            while a and sell and sell[0][0] <= p:
                q, y = heapq.heappop(sell)
                k = min(a, y)
                print(f"   khớp {k} với lệnh bán giá {q}")
                a -= k
                if y > k:
                    heapq.heappush(sell, (q, y - k))
            if a:
                heapq.heappush(buy, (-p, a)); print(f"   {a} lệnh mua vào backlog")
        else:
            while a and buy and -buy[0][0] >= p:
                nq, y = heapq.heappop(buy)
                k = min(a, y)
                print(f"   khớp {k} với lệnh mua giá {-nq}")
                a -= k
                if y > k:
                    heapq.heappush(buy, (nq, y - k))
            if a:
                heapq.heappush(sell, (p, a)); print(f"   {a} lệnh bán vào backlog")
        print(f"   buy={sorted((-x, y) for x, y in buy)}, sell={sorted(sell)}")
    return sum(y for _, y in buy + sell) % MOD

print("=>", backlog_heaps_demo(examples[0][0]))

Lô MUA giá 10 x5
   5 lệnh mua vào backlog
   buy=[(10, 5)], sell=[]
Lô BÁN giá 15 x2
   2 lệnh bán vào backlog
   buy=[(10, 5)], sell=[(15, 2)]
Lô BÁN giá 25 x1
   1 lệnh bán vào backlog
   buy=[(10, 5)], sell=[(15, 2), (25, 1)]
Lô MUA giá 30 x4
   khớp 2 với lệnh bán giá 15
   khớp 1 với lệnh bán giá 25
   1 lệnh mua vào backlog
   buy=[(10, 5), (30, 1)], sell=[]
=> 6


In [10]:
class SolutionHeap:
    def getNumberOfBacklogOrders(self, orders: List[List[int]]) -> int:
        buy, sell = [], []
        for p, a, t in orders:
            if t == 0:
                while a and sell and sell[0][0] <= p:
                    q, y = heapq.heappop(sell)
                    if a >= y:
                        a -= y
                    else:
                        heapq.heappush(sell, (q, y - a))
                        a = 0
                if a:
                    heapq.heappush(buy, (-p, a))
            else:
                while a and buy and -buy[0][0] >= p:
                    q, y = heapq.heappop(buy)
                    if a >= y:
                        a -= y
                    else:
                        heapq.heappush(buy, (q, y - a))
                        a = 0
                if a:
                    heapq.heappush(sell, (p, a))
        return sum(y for _, y in buy + sell) % MOD

for orders, want in examples:
    print(SolutionHeap().getNumberOfBacklogOrders(orders), want)

6 6
999999984 999999984


**Độ phức tạp:** mỗi lô được push tối đa 1 lần (cộng tối đa 1 lần push lại khi khớp một phần — chỉ xảy ra ở cuối vòng `while` của mỗi lô đến). Mỗi pop xóa hẳn một lô, hoặc là lần cuối của vòng ⇒ tổng `O(n)` thao tác heap, mỗi thao tác `O(log n)` ⇒ **`O(n log n)`** thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Hai heap, sửa đỉnh tại chỗ khi khớp một phần

Khi lô đến bị hết trước (`a < y`), thay vì pop rồi push `(q, y − a)`, ta **giảm số lượng ở đỉnh** (giá không đổi ⇒ heap vẫn hợp lệ, xem 3.3). Dùng list `[price, amount]` thay tuple.

In [11]:
class SolutionHeapInplace:
    def getNumberOfBacklogOrders(self, orders: List[List[int]]) -> int:
        buy, sell = [], []                 # phần tử là list có thể sửa
        for p, a, t in orders:
            same, other = (buy, sell) if t == 0 else (sell, buy)
            # với lệnh mua: other = sell, khớp khi sell[0][0] <= p
            # với lệnh bán: other = buy (lưu -price), khớp khi -buy[0][0] >= p  <=>  buy[0][0] <= -p
            key = p if t == 0 else -p
            while a and other and other[0][0] <= key:
                if other[0][1] > a:
                    other[0][1] -= a       # sửa đỉnh tại chỗ, không cần sift
                    a = 0
                else:
                    a -= heapq.heappop(other)[1]
            if a:
                heapq.heappush(same, [-p if t == 0 else p, a])
        return (sum(x[1] for x in buy) + sum(x[1] for x in sell)) % MOD

for orders, want in examples:
    print(SolutionHeapInplace().getNumberOfBacklogOrders(orders), want)

6 6
999999984 999999984


Mẹo gộp hai nhánh: với lệnh **bán** giá `p`, điều kiện `−buy[0][0] ≥ p` tương đương `buy[0][0] ≤ −p`, cùng dạng `other[0][0] ≤ key` như lệnh mua. Nhờ vậy một vòng `while` phục vụ cả hai loại lệnh; đổi lại code khó đọc hơn một chút, nên bản nộp ở mục 9 vẫn tách hai nhánh cho rõ.

**Độ phức tạp:** vẫn `O(n log n)`, nhưng bớt một cặp push/pop mỗi lần khớp một phần.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 2 `[[7,10⁹,1],[15,3,0],[5,999999995,0],[5,1,1]]`:

| lô | điều kiện khớp | kết quả | `buy` (giá×SL) | `sell` (giá×SL) |
|---|---|---|---|---|
| bán 7×10⁹ | `buy` rỗng | vào backlog | — | 7×10⁹ |
| mua 15×3 | `sell[0]=7 ≤ 15` | khớp 3 (khớp một phần lô bán) | — | 7×(10⁹−3) |
| mua 5×999999995 | `7 > 5` | vào backlog | 5×999999995 | 7×(10⁹−3) |
| bán 5×1 | `max buy = 5 ≥ 5` | khớp 1 | 5×999999994 | 7×(10⁹−3) |

Tổng `1999999991 mod (10⁹+7) = 999999984`. ✔ (Chú ý bước cuối: giá **bằng nhau** vẫn khớp.)

In [12]:
def trace_backlog(orders):
    buy, sell = [], []
    for step, (p, a, t) in enumerate(orders, 1):
        before = a
        matched = []
        if t == 0:
            while a and sell and sell[0][0] <= p:
                q, y = heapq.heappop(sell); k = min(a, y); a -= k; matched.append((q, k))
                if y > k: heapq.heappush(sell, (q, y - k))
            if a: heapq.heappush(buy, (-p, a))
        else:
            while a and buy and -buy[0][0] >= p:
                nq, y = heapq.heappop(buy); k = min(a, y); a -= k; matched.append((-nq, k))
                if y > k: heapq.heappush(buy, (nq, y - k))
            if a: heapq.heappush(sell, (p, a))
        print(f"bước {step}: {'MUA' if t == 0 else 'BÁN'} {p}x{before} | khớp {matched} | còn vào backlog {a}")
        print(f"         buy={sorted(((-x, y) for x, y in buy), reverse=True)} sell={sorted(sell)}")
    total = sum(y for _, y in buy + sell)
    print("tổng =", total, "-> mod =", total % MOD)
    return total % MOD

trace_backlog(examples[1][0])

bước 1: BÁN 7x1000000000 | khớp [] | còn vào backlog 1000000000
         buy=[] sell=[(7, 1000000000)]
bước 2: MUA 15x3 | khớp [(7, 3)] | còn vào backlog 0
         buy=[] sell=[(7, 999999997)]
bước 3: MUA 5x999999995 | khớp [] | còn vào backlog 999999995
         buy=[(5, 999999995)] sell=[(7, 999999997)]
bước 4: BÁN 5x1 | khớp [(5, 1)] | còn vào backlog 0
         buy=[(5, 999999994)] sell=[(7, 999999997)]
tổng = 1999999991 -> mod = 999999984


999999984

## 9. Code nộp LeetCode

In [13]:
import heapq
from typing import List

class Solution:
    def getNumberOfBacklogOrders(self, orders: List[List[int]]) -> int:
        buy = []    # max-heap theo giá: (-price, amount)
        sell = []   # min-heap theo giá: (price, amount)
        for price, amount, kind in orders:
            if kind == 0:                                   # lô MUA
                while amount and sell and sell[0][0] <= price:
                    p, a = heapq.heappop(sell)
                    if amount >= a:
                        amount -= a
                    else:
                        heapq.heappush(sell, (p, a - amount))
                        amount = 0
                if amount:
                    heapq.heappush(buy, (-price, amount))
            else:                                           # lô BÁN
                while amount and buy and -buy[0][0] >= price:
                    p, a = heapq.heappop(buy)
                    if amount >= a:
                        amount -= a
                    else:
                        heapq.heappush(buy, (p, a - amount))
                        amount = 0
                if amount:
                    heapq.heappush(sell, (price, amount))
        return (sum(a for _, a in buy) + sum(a for _, a in sell)) % (10**9 + 7)

for orders, want in examples:
    assert Solution().getNumberOfBacklogOrders(orders) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Tương đương với mô phỏng từng đơn vị.** Trong mô phỏng gốc, `amount` lệnh của một lô được đặt lần lượt. Xét lô mua giá `p`:
- Lệnh đơn vị đầu tiên khớp với lệnh bán rẻ nhất `q₁` (nếu `q₁ ≤ p`). Lệnh thứ hai lại khớp với lệnh bán rẻ nhất **hiện tại** — vẫn là cùng mức giá `q₁` nếu lô bán đó còn, hoặc mức kế tiếp. Vậy chuỗi khớp là: lấy lần lượt các lệnh bán theo giá tăng dần, **gom theo lô** thì đúng là `min(a, y)` mỗi lần.
- Khi gặp lệnh bán rẻ nhất `> p`, mọi lệnh đơn vị còn lại của lô **đều** không khớp (backlog bán không đổi giữa chúng) ⇒ cả phần còn lại vào backlog mua cùng lúc.

Lô bán đối xứng. Hai heap trả về đúng "rẻ nhất"/"đắt nhất" nên mỗi bước giống hệt mô phỏng gốc.

**Một chi tiết:** có thể có nhiều lô cùng giá trong một heap (ví dụ hai lô bán giá 7). Thứ tự giữa chúng không ảnh hưởng tới **tổng số lượng** còn lại, nên đáp án không đổi.

**Invariant `max(buy) < min(sell)`** giữ được sau mỗi lô: lô mua chỉ vào backlog khi `min(sell) > p` (hoặc `sell` rỗng), và mọi lệnh mua cũ vốn đã `< min(sell)`; tương tự cho lô bán.

## 11. Phân tích Time / Space

Gọi `n = len(orders)`, `T = Σ amount`.

| Cách | Time | Space |
|---|---|---|
| 1. Từng đơn vị + list | `O(T²)` | `O(T)` |
| 2. Theo lô + list | `O(n²)` | `O(n)` |
| 3. Hai heap | `O(n log n)` | `O(n)` |
| 4. Hai heap sửa đỉnh | `O(n log n)` (ít thao tác hơn) | `O(n)` |

Phân tích amortized của Cách 3: mỗi lô gây ra ≤ 1 push "vào backlog" và ≤ 1 push "trả phần dư". Mỗi pop hoặc xóa hẳn một lô (tổng ≤ số push) ⇒ ≤ `2n` push, ≤ `2n` pop.

In [14]:
rng = random.Random(1801)
big = [[rng.randint(1, 10**9), rng.randint(1, 10**9), rng.randint(0, 1)] for _ in range(10**5)]
narrow = [[rng.randint(1, 50), rng.randint(1, 10**9), rng.randint(0, 1)] for _ in range(10**5)]
for name, sol in [("Cách 3", SolutionHeap()), ("Cách 4", SolutionHeapInplace()), ("Nộp", Solution())]:
    for label, data in [("giá rộng", big), ("giá hẹp 1..50", narrow)]:
        t = time.perf_counter()
        r = sol.getNumberOfBacklogOrders([o[:] for o in data])
        print(f"{name} ({label}): {r} trong {1000*(time.perf_counter()-t):.0f} ms")
small = [o[:] for o in big[:3000]]
t = time.perf_counter(); backlog_batch_list(small); dt = time.perf_counter() - t
print(f"Cách 2 với n=3000: {dt:.2f}s")

Cách 3 (giá rộng): 409889028 trong 86 ms
Cách 3 (giá hẹp 1..50): 889133481 trong 94 ms
Cách 4 (giá rộng): 409889028 trong 58 ms
Cách 4 (giá hẹp 1..50): 889133481 trong 56 ms


Nộp (giá rộng): 409889028 trong 94 ms
Nộp (giá hẹp 1..50): 889133481 trong 61 ms
Cách 2 với n=3000: 0.02s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu mô phỏng từng đơn vị)

In [15]:
sols = [Solution(), SolutionHeap(), SolutionHeapInplace()]
edge = [
    ([[10, 1, 0]], 1),                             # một lô
    ([[10, 1, 0], [10, 1, 1]], 0),                 # giá bằng nhau: khớp
    ([[10, 1, 0], [11, 1, 1]], 2),                 # bán đắt hơn mua: không khớp
    ([[10, 1, 0], [20, 1, 0], [15, 1, 1]], 1),     # bán khớp với mua ĐẮT NHẤT (20)
    ([[10, 5, 1], [12, 2, 0]], 3),                 # khớp một phần lô bán
    ([[5, 3, 1], [5, 3, 0]], 0),
    ([[1, 4, 0], [2, 4, 0], [1, 8, 1]], 0),
]
for orders, want in examples + edge:
    for s in sols:
        assert s.getNumberOfBacklogOrders([o[:] for o in orders]) == want, (orders, s)
    assert backlog_batch_list([o[:] for o in orders]) == want
    if sum(o[1] for o in orders) <= 2000:
        assert backlog_brute(orders) == want
    print("OK", orders[:4], "->", want)

OK [[10, 5, 0], [15, 2, 1], [25, 1, 1], [30, 4, 0]] -> 6
OK [[7, 1000000000, 1], [15, 3, 0], [5, 999999995, 0], [5, 1, 1]] -> 999999984
OK [[10, 1, 0]] -> 1
OK [[10, 1, 0], [10, 1, 1]] -> 0
OK [[10, 1, 0], [11, 1, 1]] -> 2
OK [[10, 1, 0], [20, 1, 0], [15, 1, 1]] -> 1
OK [[10, 5, 1], [12, 2, 0]] -> 3
OK [[5, 3, 1], [5, 3, 0]] -> 0
OK [[1, 4, 0], [2, 4, 0], [1, 8, 1]] -> 0


### 12.2 Ngẫu nhiên có seed: so với mô phỏng từng đơn vị

Ràng buộc thu nhỏ: `n ≤ 12`, `amount ≤ 6`, giá trong `1..8` (giá nhỏ để có nhiều lần khớp).

In [16]:
rng = random.Random(99)
for _ in range(1000):
    n = rng.randint(1, 12)
    orders = [[rng.randint(1, 8), rng.randint(1, 6), rng.randint(0, 1)] for _ in range(n)]
    want = backlog_brute(orders)
    for s in sols:
        assert s.getNumberOfBacklogOrders([o[:] for o in orders]) == want, orders
    assert backlog_batch_list([o[:] for o in orders]) == want
for _ in range(30):
    orders = [[rng.randint(1, 30), rng.randint(1, 10**9), rng.randint(0, 1)] for _ in range(rng.randint(1, 400))]
    res = {s.getNumberOfBacklogOrders([o[:] for o in orders]) for s in sols}
    res.add(backlog_batch_list([o[:] for o in orders]))
    assert len(res) == 1
print("1000 case nhỏ (từng đơn vị) + 30 case amount lớn (so Cách 2): khớp")

1000 case nhỏ (từng đơn vị) + 30 case amount lớn (so Cách 2): khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên đổi dấu: dùng min-heap cho lệnh mua

In [17]:
def wrong_no_negate(orders):
    buy, sell = [], []
    for p, a, t in orders:
        if t == 0:
            while a and sell and sell[0][0] <= p:
                q, y = heapq.heappop(sell)
                if a >= y: a -= y
                else: heapq.heappush(sell, (q, y - a)); a = 0
            if a: heapq.heappush(buy, (p, a))              # SAI: buy[0] là giá NHỎ nhất
        else:
            while a and buy and buy[0][0] >= p:
                q, y = heapq.heappop(buy)
                if a >= y: a -= y
                else: heapq.heappush(buy, (q, y - a)); a = 0
            if a: heapq.heappush(sell, (p, a))
    return sum(y for _, y in buy + sell) % MOD

orders = [[10, 1, 0], [20, 1, 0], [15, 1, 1]]
print("sai:", wrong_no_negate(orders), "| đúng:", Solution().getNumberOfBacklogOrders(orders))

sai: 3 | đúng: 1


**Vì sao sai:** lệnh bán giá 15 phải nhìn lệnh mua **đắt nhất** (20 ≥ 15 ⇒ khớp). Min-heap đưa ra 10 < 15 ⇒ tưởng không khớp. **Sửa:** lưu `(-price, amount)` cho `buy`.

### Lỗi 2 — So sánh nghiêm ngặt `<` thay vì `<=`

In [18]:
def wrong_strict(orders):
    buy, sell = [], []
    for p, a, t in orders:
        if t == 0:
            while a and sell and sell[0][0] < p:           # SAI: giá bằng nhau cũng khớp
                q, y = heapq.heappop(sell)
                if a >= y: a -= y
                else: heapq.heappush(sell, (q, y - a)); a = 0
            if a: heapq.heappush(buy, (-p, a))
        else:
            while a and buy and -buy[0][0] > p:            # SAI
                q, y = heapq.heappop(buy)
                if a >= y: a -= y
                else: heapq.heappush(buy, (q, y - a)); a = 0
            if a: heapq.heappush(sell, (p, a))
    return sum(y for _, y in buy + sell) % MOD

for orders, want in [([[10, 1, 0], [10, 1, 1]], 0), examples[1]]:
    print("sai:", wrong_strict(orders), "| đúng:", want)

sai: 2 | đúng: 0
sai: 999999986 | đúng: 999999984


**Vì sao sai:** đề viết "smaller than **or equal to**" / "larger than **or equal to**". Ví dụ 2 bước cuối (bán 5 gặp mua 5) chính là bẫy này.

### Lỗi 3 — Khớp một phần nhưng quên đẩy phần dư lại heap

In [19]:
def wrong_lose_rest(orders):
    buy, sell = [], []
    for p, a, t in orders:
        if t == 0:
            while a and sell and sell[0][0] <= p:
                q, y = heapq.heappop(sell)
                a = max(a - y, 0)                          # SAI: phần y - a của lô bán biến mất
            if a: heapq.heappush(buy, (-p, a))
        else:
            while a and buy and -buy[0][0] >= p:
                q, y = heapq.heappop(buy)
                a = max(a - y, 0)
            if a: heapq.heappush(sell, (p, a))
    return sum(y for _, y in buy + sell) % MOD

orders = [[10, 5, 1], [12, 2, 0]]
print("sai:", wrong_lose_rest(orders), "| đúng:", Solution().getNumberOfBacklogOrders(orders))

sai: 0 | đúng: 3


**Vì sao sai:** lô bán 10×5 chỉ bị khớp 2, còn 3 lệnh phải ở lại backlog. Pop ra mà không push lại `(10, 3)` là làm mất lệnh. **Sửa:** nhánh `else: heappush(..., (q, y - a))`.

### Lỗi 4 — Khớp từng đơn vị một (vòng lặp theo `amount`)

In [20]:
def slow_unit_by_unit(orders):
    buy, sell = [], []
    for p, a, t in orders:
        for _ in range(a):                                 # CHẬM: amount tới 10^9
            if t == 0:
                if sell and sell[0][0] <= p:
                    q, y = heapq.heappop(sell)
                    if y > 1: heapq.heappush(sell, (q, y - 1))
                else:
                    heapq.heappush(buy, (-p, 1))
            else:
                if buy and -buy[0][0] >= p:
                    q, y = heapq.heappop(buy)
                    if y > 1: heapq.heappush(buy, (q, y - 1))
                else:
                    heapq.heappush(sell, (p, 1))
    return sum(y for _, y in buy + sell) % MOD

test = [[7, 20000, 1], [15, 3, 0], [5, 19995, 0], [5, 1, 1]]     # Ví dụ 2 thu nhỏ
t = time.perf_counter(); r = slow_unit_by_unit(test); dt = time.perf_counter() - t
print(f"kết quả {r} (đúng: {Solution().getNumberOfBacklogOrders(test)}) trong {dt*1000:.0f} ms cho T=4·10^4")
print(f"ước tính cho Ví dụ 2 thật (T≈2·10^9): ≈ {dt * 2e9 / 4e4 / 60:.0f} phút")

kết quả 39991 (đúng: 39991) trong 6 ms cho T=4·10^4
ước tính cho Ví dụ 2 thật (T≈2·10^9): ≈ 5 phút


**Vì sao sai:** đúng về logic nhưng `amount` tới `10⁹` ⇒ TLE. **Sửa:** khớp `min(a, y)` một lần.

### Lỗi 5 — Quên modulo ở kết quả

In [21]:
orders = examples[1][0]
raw = sum(o[1] for o in [[7, 10**9 - 3, 1], [5, 999999994, 0]])
print("không mod:", raw, "| đề yêu cầu:", Solution().getNumberOfBacklogOrders(orders))

không mod: 1999999991 | đề yêu cầu: 999999984


**Vì sao sai:** tổng có thể vượt `10⁹ + 7` (ở đây `1999999991`). Python không tràn số nên dễ quên. **Sửa:** `% (10**9 + 7)` ở **kết quả**, không ở số lượng trung gian.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Luôn lấy phần tử nhỏ nhất / lớn nhất hiện có, có thêm/bớt liên tục"** ⇒ heap.
- **Hai phía đối nghịch** (mua/bán, trái/phải) ⇒ **hai heap**: một min, một max (đổi dấu).
- **Số lượng lớn theo lô** ⇒ lưu `(khóa, số lượng)` và khớp `min(a, y)`, tuyệt đối không lặp từng đơn vị.
- **Khớp một phần** ⇒ nhớ trả phần dư (push lại hoặc sửa đỉnh tại chỗ khi khóa không đổi).
- Modulo chỉ khi **trả kết quả** nếu giá trị trung gian còn dùng để so sánh.

## 15. Biến thể của bài

1. **Trả thêm khối lượng/giá trị giao dịch:** tổng số lệnh đã khớp, hoặc tổng tiền nếu khớp ở giá của lệnh trong backlog (Bài tập 5).
2. **Hủy lệnh** (cancel) theo id ⇒ heap + "xóa lười" (lazy deletion) hoặc cây cân bằng (`SortedList`).
3. **Khớp theo thời gian ưu tiên giá–thời gian** (price-time priority): thêm thời điểm vào khóa `(price, time, amount)`.
4. **Truy vấn backlog giữa chừng** (ví dụ "có bao nhiêu lệnh mua giá ≥ x") ⇒ cây Fenwick trên giá đã nén.
5. Bài cùng mẫu hai heap: LeetCode 295 *Find Median from Data Stream*, 502 *IPO*, 1834 *Single-Threaded CPU*.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng đơn vị + list | đúng câu chữ đề | `O(T²)` | `O(T)` | kiểm chứng, `T` nhỏ |
| 2. Theo lô + list | khớp `min(a,y)`, quét min/max | `O(n²)` | `O(n)` | `n` nhỏ |
| 3. Hai heap | min-heap bán, max-heap mua | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. Hai heap sửa đỉnh | như 3, không pop/push khi khớp một phần | `O(n log n)` | `O(n)` | tối ưu hằng số |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `orders = [[5,3,1],[5,3,0]]` và `orders = [[3,2,0],[4,2,1],[4,5,0]]` cho kết quả bao nhiêu?

**Bài 2 (tính tay):** Lập bảng trạng thái hai heap cho Ví dụ 1 (đã có ở mục 4) — tự làm lại rồi so với `trace_backlog`.

**Bài 3 (điền TODO):** Viết `backlog_split(orders)` trả về **bộ hai số** `(tổng lệnh mua, tổng lệnh bán)` còn trong backlog (không lấy modulo).

**Bài 4 (tự viết lại):** Viết lại lời giải dùng **một hàm phụ** `match(heap, amount, ok)` dùng chung cho hai nhánh mua/bán, trong đó `ok(top_key)` cho biết đỉnh có khớp được không.

**Bài 5 (mini challenge):** Mỗi lần khớp, giao dịch diễn ra ở **giá của lệnh đang nằm trong backlog**. Viết `traded_value(orders)` trả về **tổng tiền** đã giao dịch (tổng `giá × số lượng khớp`), không modulo.

In [22]:
def backlog_split(orders):
    # TODO
    pass

def solve_with_match(orders):
    # TODO: dùng hàm match chung
    pass

def traded_value(orders):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Giá bằng nhau thì khớp | Bài thứ hai: mua 3×2, bán 4×2 không khớp | Mua 4×5 khớp 2 lệnh bán giá 4, còn 3 lệnh mua 4 + 2 lệnh mua 3 |
| 2 | Dùng bảng ở mục 4 | Chú ý lô cuối khớp 2 lần | `trace_backlog(examples[0][0])` |
| 3 | Giữ nguyên Cách 3 | Tính hai tổng riêng ở cuối | `sum(a for _, a in buy), sum(a for _, a in sell)` |
| 4 | `match` trả về `amount` còn lại | Heap mua lưu `-price` nên `ok = lambda k: -k >= p` | Heap bán: `ok = lambda k: k <= p` |
| 5 | Mỗi lần khớp `k` lệnh với đỉnh giá `q` | Cộng `q * k` | Đối chiếu với mô phỏng từng đơn vị |

## 19. Đáp án tham khảo

In [23]:
r1 = [Solution().getNumberOfBacklogOrders(o) for o in ([[5, 3, 1], [5, 3, 0]], [[3, 2, 0], [4, 2, 1], [4, 5, 0]])]
print("Bài 1:", r1)
assert r1 == [0, 5]
print("Bài 2:"); assert trace_backlog(examples[0][0]) == 6

Bài 1: [0, 5]
Bài 2:
bước 1: MUA 10x5 | khớp [] | còn vào backlog 5
         buy=[(10, 5)] sell=[]
bước 2: BÁN 15x2 | khớp [] | còn vào backlog 2
         buy=[(10, 5)] sell=[(15, 2)]
bước 3: BÁN 25x1 | khớp [] | còn vào backlog 1
         buy=[(10, 5)] sell=[(15, 2), (25, 1)]
bước 4: MUA 30x4 | khớp [(15, 2), (25, 1)] | còn vào backlog 1
         buy=[(30, 1), (10, 5)] sell=[]
tổng = 6 -> mod = 6


In [24]:
def match(heap, amount, ok):
    while amount and heap and ok(heap[0][0]):
        key, y = heapq.heappop(heap)
        if amount >= y:
            amount -= y
        else:
            heapq.heappush(heap, (key, y - amount))
            amount = 0
    return amount

def _run(orders):
    buy, sell = [], []
    for p, a, t in orders:
        if t == 0:
            a = match(sell, a, lambda k: k <= p)
            if a: heapq.heappush(buy, (-p, a))
        else:
            a = match(buy, a, lambda k: -k >= p)
            if a: heapq.heappush(sell, (p, a))
    return buy, sell

def backlog_split(orders):
    buy, sell = _run(orders)
    return sum(a for _, a in buy), sum(a for _, a in sell)

def solve_with_match(orders):
    return sum(backlog_split(orders)) % MOD

print("Bài 3:", backlog_split(examples[1][0]))
assert backlog_split(examples[1][0]) == (999999994, 999999997)
rng = random.Random(5)
for _ in range(300):
    orders = [[rng.randint(1, 6), rng.randint(1, 5), rng.randint(0, 1)] for _ in range(rng.randint(1, 10))]
    assert solve_with_match(orders) == backlog_brute(orders)
print("Bài 4: OK")

Bài 3: (999999994, 999999997)
Bài 4: OK


In [25]:
def traded_value(orders):
    buy, sell = [], []
    value = 0
    for p, a, t in orders:
        if t == 0:
            while a and sell and sell[0][0] <= p:
                q, y = heapq.heappop(sell); k = min(a, y); a -= k; value += q * k
                if y > k: heapq.heappush(sell, (q, y - k))
            if a: heapq.heappush(buy, (-p, a))
        else:
            while a and buy and -buy[0][0] >= p:
                nq, y = heapq.heappop(buy); k = min(a, y); a -= k; value += -nq * k
                if y > k: heapq.heappush(buy, (nq, y - k))
            if a: heapq.heappush(sell, (p, a))
    return value

def traded_value_brute(orders):
    buy, sell, value = [], [], 0
    for p, a, t in orders:
        for _ in range(a):
            if t == 0:
                if sell and min(sell) <= p:
                    q = min(sell); sell.remove(q); value += q
                else:
                    buy.append(p)
            else:
                if buy and max(buy) >= p:
                    q = max(buy); buy.remove(q); value += q
                else:
                    sell.append(p)
    return value

print("Bài 5, Ví dụ 1:", traded_value(examples[0][0]), "(= 2·15 + 1·25)")
assert traded_value(examples[0][0]) == 55
rng = random.Random(12)
for _ in range(400):
    orders = [[rng.randint(1, 8), rng.randint(1, 5), rng.randint(0, 1)] for _ in range(rng.randint(1, 10))]
    assert traded_value(orders) == traded_value_brute(orders), orders
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK so với mô phỏng từng đơn vị")

Bài 5, Ví dụ 1: 55 (= 2·15 + 1·25)
Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK so với mô phỏng từng đơn vị


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
buy, sell = [], []                    # buy: (-price, amt) max-heap; sell: (price, amt) min-heap
for p, a, t in orders:
    if t == 0:
        while a and sell and sell[0][0] <= p:
            q, y = heappop(sell)
            if a >= y: a -= y
            else: heappush(sell, (q, y - a)); a = 0
        if a: heappush(buy, (-p, a))
    else:                             # đối xứng với -buy[0][0] >= p
        ...
return (sum(a for _, a in buy) + sum(a for _, a in sell)) % MOD
```

**Checklist:**
- [ ] `buy` lưu `-price` (max-heap), `sell` lưu `price`?
- [ ] So sánh `<=` / `>=` (giá bằng nhau vẫn khớp)?
- [ ] Khớp theo **lô** `min(a, y)`, không lặp từng đơn vị?
- [ ] Khớp một phần thì **trả phần dư** về heap?
- [ ] Modulo ở **kết quả cuối**?